Check the implementation of various models

In [120]:
from reload import reload_package
reload_package(namespace=globals())

Reloaded 3 modules, rebound 3 names.


In [1]:
import altair as alt
import numpy as np
import polars as pl
import torch

alt.renderers.enable("jupyter", offline=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Data

In [9]:
from ts_uncertainty.plotting import plot_sample
from ts_uncertainty.synthetic import SyntheticGenerator

In [10]:
context_length = 64
horizon_length = 12

sin_gen = SyntheticGenerator(
    context_length=context_length,
    horizon=horizon_length,
    seasonal=True,
    covariate=True,
    periods=(24,),             # period of the sine
    n_periods=1,
    n_harmonics=1,             # 1 harmonic = pure sine, no extra shape
    level_range=(0.0, 0.0),    # centered at 0
    slope_range=(0.0, 0.0),    # no trend
    amplitude_range=(1.0, 1.0),
)
s = sin_gen.sample()
plot_sample(s, title="Sin Sample")

# Statsforecasting

In [122]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA
from ts_uncertainty.sample import samples_to_sf_inputs

In [123]:
# model input and output shapes
x, c, y = samples_to_sf_inputs([s])

In [124]:
sf = StatsForecast(
    models=[AutoARIMA(season_length=1)],
    freq=1,
)
fc = sf.forecast(df=x, h=horizon_length, X_df=c, level=[90])
point_fc = fc["AutoARIMA"].to_numpy()
plot_sample(s, title="Sin Sample", prediction=point_fc, quantiles=None, coverages=None, width=800)

# Chronos2

In [2]:
from chronos import Chronos2Pipeline, BaseChronosPipeline
from ts_uncertainty.sample import samples_to_chronos2_inputs

In [3]:
# model input and output shapes
x, y = samples_to_chronos2_inputs([s])

NameError: name 's' is not defined

In [127]:
# loading model
chronos2_pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map=device)
chronos2_config = chronos2_pipeline.model.chronos_config
print(chronos2_config)

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Chronos2ForecastingConfig(context_length=8192, output_patch_size=16, input_patch_size=16, input_patch_stride=16, quantiles=[0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.99], use_reg_token=True, use_arcsinh=True, max_output_patches=64, time_encoding_scale=8192)


In [128]:
# predict
chronos2_out = chronos2_pipeline.predict(x, prediction_length=horizon_length, cross_learning=False)
print (chronos2_out[0].shape)
plot_sample(s, title="Sin Sample", prediction=chronos2_out[0], quantiles=chronos2_config.quantiles, coverages=[0.8], width=800)

torch.Size([1, 21, 12])


# TiRex-2

In [4]:
from tirex2 import load_model
from ts_uncertainty.sample import samples_to_tirex2_inputs

In [11]:
# model input and output shapes
x, y = samples_to_tirex2_inputs([s])

In [6]:
# loading model
model = load_model("NX-AI/TiRex-2", device=device)
tirex2_config = model.model.stack_config 
print(tirex2_config)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

MultivariateStackConfig(templates={'slstm_attn': MultivariateBlockConfig(time_mixer=TimeMixerConfig(model_type='bi-slstm', act_fn=SiLU(), embedding_dim=512, num_heads=4, num_slstm_heads=4, device='cuda'), variate_mixer=VariateMixerConfig(act_fn=SiLU(), embedding_dim=512, num_heads=4, use_qk_norm=True, use_flex_attention=False), block_order=None, block_idx=None, num_blocks=None, dropout=0.1, eps=1e-07), 'mlstm_attn': MultivariateBlockConfig(time_mixer=TimeMixerConfig(model_type='bi-mlstm', act_fn=SiLU(), embedding_dim=512, num_heads=4, num_slstm_heads=None, device='cuda'), variate_mixer=VariateMixerConfig(act_fn=SiLU(), embedding_dim=512, num_heads=4, use_qk_norm=True, use_flex_attention=False), block_order=None, block_idx=None, num_blocks=None, dropout=0.1, eps=1e-07)}, recipe=['mlstm_attn', 'slstm_attn', 'mlstm_attn', 'slstm_attn', 'mlstm_attn', 'slstm_attn', 'mlstm_attn', 'slstm_attn', 'mlstm_attn', 'slstm_attn', 'mlstm_attn', 'slstm_attn'])


In [14]:
model.model.quantiles

tensor([0.1000, 0.2000, 0.3000, 0.4000, 0.5000, 0.6000, 0.7000, 0.8000, 0.9000],
       device='cuda:0')

In [132]:
# predict
tirex2_out = model.predict(x, prediction_length=horizon_length)
print (tirex2_out[0].shape)
plot_sample(s, title="Sin Sample", prediction=tirex2_out[0], quantiles=model.model.quantiles, coverages=[0.8], width=800)

torch.Size([1, 9, 12])


# Timestral

In [15]:
from timestral import TimestralForecaster
from ts_uncertainty.sample import samples_to_timestral_inputs

In [16]:
# model input and output shapes
context, past, known, y = samples_to_timestral_inputs([s])

In [17]:
# loading model
model = TimestralForecaster.from_pretrained("mistralai/Timestral-209M", device=device)
timestral_config = model.config
print(timestral_config)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

TimestralConfig(patch_size=32, d_model=768, num_heads=12, mlp_hidden_dim=2048, block_schedule=('time', 'time', 'time', 'time_variate', 'time', 'time', 'time', 'time_variate', 'time', 'time', 'time', 'time_variate', 'time', 'time', 'time', 'time_variate', 'time', 'time', 'time', 'time_variate', 'time', 'time', 'time', 'time_variate'), quantile_knots=(0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.99), block_size=32, context_length=32768, rope_theta=10000.0, rope_partial_factor=0.5, norm_eps=1e-06, attn_bias=False, mlp_bias=False, time_attn_gate='sdpa_sigmoid', variate_attn_gate='sdpa_sigmoid', minimum_scale=1e-06, use_asinh=True, stabilize_with_global=False, covariate_enabled=True, split_context_horizon_attn=True, attend_missing_patches=True, quantile_width_factor=3.0)


In [136]:
# predict
timestral_out = model.forecast(context, prediction_length=horizon_length, past_covariates=past, known_covariates=known)
print (timestral_out.quantiles[0].shape)
plot_sample(s, title="Sin Sample", prediction=timestral_out.quantiles[0].transpose(2,1), quantiles=model.config.quantile_knots, coverages=[0.8], width=800)

torch.Size([1, 12, 21])
